# DriveZA Holdings Fleet Database
Creates the warehouse, database, schema, and four fleet tables in Snowflake.

In [ ]:
%%sql -r wh_result
CREATE WAREHOUSE IF NOT EXISTS DRZ_WH
    WAREHOUSE_SIZE = 'X-SMALL'
    AUTO_SUSPEND   = 60
    AUTO_RESUME    = TRUE
    COMMENT        = 'DriveZA warehouse auto-suspends after 60s';

In [ ]:
%%sql -r db_result
CREATE DATABASE IF NOT EXISTS DRIVEZA_FLEET;

In [ ]:
%%sql -r use_db_result
USE DATABASE DRIVEZA_FLEET;
USE WAREHOUSE DRZ_WH;

In [ ]:
%%sql -r schema_result
CREATE SCHEMA IF NOT EXISTS DRIVEZA_FLEET.fleet;
USE SCHEMA DRIVEZA_FLEET.fleet;

In [ ]:
%%sql -r branches_result
CREATE OR REPLACE TABLE DRIVEZA_FLEET.fleet.branches (
    branch_id           VARCHAR(10)         NOT NULL,
    branch_name         VARCHAR(100)        NOT NULL,
    province            VARCHAR(50)         NOT NULL,
    city                VARCHAR(50)         NOT NULL,
    address             VARCHAR(150)        NOT NULL,
    postal_code         VARCHAR(10)         NOT NULL,
    phone               VARCHAR(20)         NOT NULL,
    email               VARCHAR(100)        NOT NULL,
    is_airport_branch   BOOLEAN             NOT NULL    DEFAULT FALSE,
    opening_hours       VARCHAR(20)         NOT NULL,
    manager_name        VARCHAR(100)        NOT NULL,
    fleet_capacity      NUMBER(5,0)         NOT NULL,
    is_active           BOOLEAN             NOT NULL    DEFAULT TRUE,
    created_at          TIMESTAMP_NTZ       NOT NULL,
    updated_at          TIMESTAMP_NTZ       NOT NULL,
    source_system       VARCHAR(30)         NOT NULL,

    CONSTRAINT pk_branches PRIMARY KEY (branch_id)
);

In [ ]:
%%sql -r vehicles_result
CREATE OR REPLACE TABLE DRIVEZA_FLEET.fleet.vehicles (
    vehicle_id              VARCHAR(10)     NOT NULL,
    vin                     VARCHAR(20)     NOT NULL,
    registration_number     VARCHAR(20)     NOT NULL,
    make                    VARCHAR(30)     NOT NULL,
    model                   VARCHAR(30)     NOT NULL,
    variant                 VARCHAR(50)     NOT NULL,
    year                    NUMBER(4,0)     NOT NULL,
    color                   VARCHAR(20)     NOT NULL,
    category                VARCHAR(20)     NOT NULL,
    fuel_type               VARCHAR(15)     NOT NULL,
    transmission            VARCHAR(15)     NOT NULL,
    seats                   NUMBER(3,0)     NOT NULL,
    doors                   NUMBER(3,0)     NOT NULL,
    engine_cc               NUMBER(6,0)     NOT NULL,
    daily_rate_zar          NUMBER(10,2)    NOT NULL,
    km_limit_per_day        VARCHAR(15)     NOT NULL,
    excess_km_rate_zar      NUMBER(8,2)     NOT NULL,
    purchase_date           DATE            NOT NULL,
    purchase_price_zar      NUMBER(12,2)    NOT NULL,
    current_odometer_km     NUMBER(10,0)    NOT NULL,
    last_service_km         NUMBER(10,0)    NOT NULL,
    next_service_km         NUMBER(10,0)    NOT NULL,
    status                  VARCHAR(25)     NOT NULL,
    branch_id               VARCHAR(10)     NOT NULL,
    insurance_expiry        DATE            NOT NULL,
    roadworthy_expiry       DATE            NOT NULL,
    tracker_installed       BOOLEAN         NOT NULL    DEFAULT FALSE,
    tracker_unit_id         VARCHAR(20)     NULL,
    created_at              TIMESTAMP_NTZ   NOT NULL,
    updated_at              TIMESTAMP_NTZ   NOT NULL,
    source_system           VARCHAR(30)     NOT NULL,

    CONSTRAINT pk_vehicles      PRIMARY KEY (vehicle_id),
    CONSTRAINT uq_vehicles_vin  UNIQUE (vin),
    CONSTRAINT uq_vehicles_reg  UNIQUE (registration_number)
);

In [ ]:
%%sql -r maintenance_result
CREATE OR REPLACE TABLE DRIVEZA_FLEET.fleet.maintenance (
    maintenance_id          VARCHAR(10)     NOT NULL,
    vehicle_id              VARCHAR(10)     NOT NULL,
    branch_id               VARCHAR(10)     NOT NULL,
    maintenance_type        VARCHAR(50)     NOT NULL,
    description             VARCHAR(200)    NOT NULL,
    scheduled_date          DATE            NOT NULL,
    completed_date          DATE            NULL,
    odometer_at_service     NUMBER(10,0)    NOT NULL,
    cost_zar                NUMBER(12,2)    NOT NULL,
    parts_cost_zar          NUMBER(12,2)    NOT NULL,
    labour_cost_zar         NUMBER(12,2)    NOT NULL,
    supplier_name           VARCHAR(100)    NOT NULL,
    technician_name         VARCHAR(100)    NOT NULL,
    status                  VARCHAR(20)     NOT NULL,
    next_service_km         NUMBER(10,0)    NOT NULL,
    warranty_claim          BOOLEAN         NOT NULL    DEFAULT FALSE,
    notes                   VARCHAR(300)    NULL,
    created_at              TIMESTAMP_NTZ   NOT NULL,
    updated_at              TIMESTAMP_NTZ   NOT NULL,
    source_system           VARCHAR(30)     NOT NULL,

    CONSTRAINT pk_maintenance PRIMARY KEY (maintenance_id)
);

In [ ]:
%%sql -r incidents_result
CREATE OR REPLACE TABLE DRIVEZA_FLEET.fleet.incidents (
    incident_id                 VARCHAR(10)     NOT NULL,
    rental_id                   VARCHAR(10)     NOT NULL,
    vehicle_id                  VARCHAR(10)     NOT NULL,
    customer_id                 VARCHAR(10)     NOT NULL,
    incident_type               VARCHAR(50)     NOT NULL,
    incident_date               DATE            NOT NULL,
    report_date                 DATE            NOT NULL,
    location_description        VARCHAR(200)    NOT NULL,
    description                 TEXT            NOT NULL,
    estimated_damage_zar        NUMBER(12,2)    NOT NULL,
    actual_repair_cost_zar      NUMBER(12,2)    NULL,
    insurance_claim_number      VARCHAR(30)     NULL,
    insurance_payout_zar        NUMBER(12,2)    NULL,
    excess_charged_zar          NUMBER(10,2)    NOT NULL,
    third_party_involved        BOOLEAN         NOT NULL    DEFAULT FALSE,
    police_report_number        VARCHAR(30)     NULL,
    status                      VARCHAR(30)     NOT NULL,
    fault                       VARCHAR(20)     NOT NULL,
    created_at                  TIMESTAMP_NTZ   NOT NULL,
    updated_at                  TIMESTAMP_NTZ   NOT NULL,
    source_system               VARCHAR(30)     NOT NULL,

    CONSTRAINT pk_incidents PRIMARY KEY (incident_id)
);

In [ ]:
%%sql -r dataframe_1
USE DATABASE DRIVEZA_FLEET;
USE SCHEMA fleet;
USE WAREHOUSE DRZ_WH;

-- ── Create an internal stage ──────────────────────────────────────────────
-- A stage is a temporary holding area inside Snowflake for files
CREATE OR REPLACE STAGE drz_fleet_stage
    FILE_FORMAT = (
        TYPE             = 'CSV'
        FIELD_OPTIONALLY_ENCLOSED_BY = '"'
        SKIP_HEADER      = 1
        NULL_IF          = ('', 'NULL', 'None', 'nan')
        EMPTY_FIELD_AS_NULL = TRUE
        DATE_FORMAT      = 'YYYY-MM-DD'
        TIMESTAMP_FORMAT = 'YYYY-MM-DDTHH:MI:SS'
    )
    COMMENT = 'Staging area for DriveZA fleet CSV uploads';